# Milestone 1: IEEE-CIS Fraud Detection Exploratory Data Analysis (EDA)

**Project**: FraudShield (redesigned)  
**Objective**: Systematic, leakage-safe exploration of raw transactions and identity records.  

### Core Principles & Caveats
1. **Never fabricate facts or assumptions**: All statistics in this notebook are computed from raw competition files.
2. **Proxy Entity Key**: `card1` (and `card1 + addr1`) serve strictly as proxy entity identifiers, not verified accounts.
3. **Relative Time**: `TransactionDT` is a relative offset in seconds, not a calendar timestamp.
4. **Anonymised Features**: V, C, D, and M columns are anonymised with unknown domain semantics.
5. **Review Prioritisation**: The system produces an *estimated risk score* to prioritise manual review.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load joined dataset
data_path = Path("../data/interim/joined_transactions.parquet")
if not data_path.exists():
    data_path = Path("data/interim/joined_transactions.parquet")

print(f"Loading joined data from {data_path}...")
df = pd.read_parquet(data_path)
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.")

## 1. Class Distribution & Imbalance

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_rate = df['isFraud'].mean() * 100

print(f"Legitimate (0): {fraud_counts[0]:,} ({100 - fraud_rate:.2f}%)")
print(f"Fraud (1):      {fraud_counts[1]:,} ({fraud_rate:.4f}%)")
print(f"Total Fraud Rate: {fraud_rate:.4f}%")

fig, ax = plt.subplots(figsize=(6, 4))
fraud_counts.plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=ax)
ax.set_title('Transaction Label Distribution (isFraud)')
ax.set_xlabel('isFraud')
ax.set_ylabel('Transaction Count')
plt.tight_layout()
plt.show()

## 2. Temporal Distribution & Ties (`TransactionDT`)

We inspect `TransactionDT` span, monotonicity, and duplicate timestamp ties.

In [ ]:
dt = df['TransactionDT']
min_dt, max_dt = dt.min(), dt.max()
span_days = (max_dt - min_dt) / 86400.0

print(f"Min TransactionDT: {min_dt:,}")
print(f"Max TransactionDT: {max_dt:,}")
print(f"Span in Days: {span_days:.2f} days (~{span_days / 30.4:.1f} months)")

# Check monotonicity and ties
is_sorted = dt.is_monotonic_increasing
unique_times = dt.nunique()
ties = len(df) - unique_times
print(f"Monotonic increasing: {is_sorted}")
print(f"Unique timestamps: {unique_times:,}")
print(f"Duplicate timestamp rows (ties): {ties:,} ({ties / len(df) * 100:.2f}%)")

# Daily transaction volume histogram (86400 seconds / bin)
fig, ax = plt.subplots(figsize=(10, 4))
((dt - min_dt) / 86400).plot(kind='hist', bins=int(span_days) + 1, color='#34495e', edgecolor='white', ax=ax)
ax.set_title('Transaction Volume over Relative Days')
ax.set_xlabel('Relative Day')
ax.set_ylabel('Transactions per Day')
plt.tight_layout()
plt.show()

## 3. Proxy Entity Key Analysis: `card1` vs `(card1, addr1)`

Because IEEE-CIS has no ground-truth user/account IDs, we evaluate whether `card1` or `card1+addr1` provide sufficient recurrence for backward-looking entity history.

In [ ]:
card1_counts = df['card1'].value_counts()
comp_counts = (df['card1'].astype(str) + '_' + df['addr1'].fillna(-1).astype(int).astype(str)).value_counts()

print(f"card1 unique entities: {len(card1_counts):,}")
print(f"card1 single-transaction entities: {(card1_counts == 1).sum():,} ({(card1_counts == 1).mean() * 100:.2f}%)")
print(f"card1 multi-transaction entities:  {(card1_counts > 1).sum():,} ({(card1_counts > 1).mean() * 100:.2f}%)")
print(f"card1 max txns: {card1_counts.max():,}, median: {card1_counts.median():.1f}, 95th pct: {card1_counts.quantile(0.95):.1f}")

print("-" * 50)
print(f"(card1, addr1) unique entities: {len(comp_counts):,}")
print(f"(card1, addr1) single-transaction: {(comp_counts == 1).sum():,} ({(comp_counts == 1).mean() * 100:.2f}%)")
print(f"(card1, addr1) multi-transaction:  {(comp_counts > 1).sum():,} ({(comp_counts > 1).mean() * 100:.2f}%)")
print(f"(card1, addr1) max txns: {comp_counts.max():,}, median: {comp_counts.median():.1f}, 95th pct: {comp_counts.quantile(0.95):.1f}")

## 4. Missing Value Distribution

In [ ]:
missing = df.isna().mean() * 100
print(f"Columns with 0% missing: {(missing == 0).sum()} of {df.shape[1]}")
print(f"Columns with >50% missing: {(missing > 50).sum()}")
print(f"Columns with >80% missing: {(missing > 80).sum()}")

fig, ax = plt.subplots(figsize=(8, 4))
missing.plot(kind='hist', bins=20, color='#8e44ad', edgecolor='white', ax=ax)
ax.set_title('Distribution of Missingness Across Columns')
ax.set_xlabel('Percentage Missing (%)')
ax.set_ylabel('Number of Columns')
plt.tight_layout()
plt.show()